# TRSP Training Analysis

Visualise training statistics for a SARSA policy saved under `runs/trsp/<policy_id>/`.

Set `POLICY_ID` below, then **Run All**.

In [ ]:
POLICY_ID = 'a7739ceb'

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

# resolve paths relative to repo root regardless of cwd
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
POLICY_DIR = os.path.join(REPO_ROOT, 'runs', 'trsp', POLICY_ID)

log_path     = os.path.join(POLICY_DIR, 'train_log.csv')
results_path = os.path.join(POLICY_DIR, 'training_results.pkl')
policy_path  = os.path.join(POLICY_DIR, 'policy.pkl')

# --- load ---
log = pd.read_csv(log_path, skipinitialspace=True)
log.columns = ['episode', 'reward', 'epsilon']

with open(results_path, 'rb') as f:
    stats = pickle.load(f)

with open(policy_path, 'rb') as f:
    Q = pickle.load(f)

print(f'Policy: {POLICY_ID}')
print(f'Episodes: {len(log)}')
print(f'Q-table states: {len(Q)}')
print(f'State-action pairs visited: {len(stats["sa_visits"])}')

## 1 — Reward per episode

In [ ]:
def smooth(x, window=50):
    return pd.Series(x).rolling(window, min_periods=1, center=True).mean().values

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ep = log['episode'].values
rew = log['reward'].values

ax = axes[0]
ax.plot(ep, rew, alpha=0.25, linewidth=0.6, color='steelblue', label='raw')
ax.plot(ep, smooth(rew, 50), linewidth=1.8, color='steelblue', label='MA-50')
ax.plot(ep, smooth(rew, 200), linewidth=2.2, color='navy', label='MA-200')
ax.axhline(0, color='k', linewidth=0.6, linestyle='--')
ax.set_ylabel('Total reward')
ax.set_title(f'Episode reward — policy {POLICY_ID}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(ep, log['epsilon'].values, color='coral', linewidth=1.5)
ax2.set_ylabel('ε (epsilon)')
ax2.set_xlabel('Episode')
ax2.set_title('Epsilon decay')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(POLICY_DIR, 'reward_curve.png'), dpi=150)
plt.show()

## 2 — Exploration: new states / state-action pairs per episode

In [ ]:
nse = np.array(stats['new_states_explored'])
nsa = np.array(stats['new_sa_explored'])
eps_idx = np.arange(1, len(nse) + 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# --- new states per episode (raw + cumulative) ---
ax = axes[0, 0]
ax.plot(eps_idx, nse, alpha=0.3, linewidth=0.6, color='seagreen')
ax.plot(eps_idx, smooth(nse, 50), linewidth=1.8, color='seagreen')
ax.set_title('New states explored / episode')
ax.set_ylabel('count')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(eps_idx, np.cumsum(nse), linewidth=1.8, color='seagreen')
ax.set_title('Cumulative unique states')
ax.set_ylabel('states')
ax.grid(True, alpha=0.3)

# --- new SA pairs per episode (raw + cumulative) ---
ax = axes[1, 0]
ax.plot(eps_idx, nsa, alpha=0.3, linewidth=0.6, color='darkorange')
ax.plot(eps_idx, smooth(nsa, 50), linewidth=1.8, color='darkorange')
ax.set_title('New (state, action) pairs explored / episode')
ax.set_xlabel('Episode')
ax.set_ylabel('count')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(eps_idx, np.cumsum(nsa), linewidth=1.8, color='darkorange')
ax.set_title('Cumulative unique (s,a) pairs')
ax.set_xlabel('Episode')
ax.set_ylabel('pairs')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(POLICY_DIR, 'exploration_curves.png'), dpi=150)
plt.show()

## 3 — State visit distribution (top-50 most visited states)

In [ ]:
sv = stats['state_visits']  # dict[(node_idx, target_idx)] -> count
top_n = 50
top_states = sorted(sv.items(), key=lambda x: -x[1])[:top_n]
labels = [str(s) for s, _ in top_states]
counts = [c for _, c in top_states]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(top_n), counts, color='mediumpurple')
ax.set_xticks(range(top_n))
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_title(f'Top-{top_n} most-visited states (node, target)')
ax.set_ylabel('Visit count')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(POLICY_DIR, 'state_visits.png'), dpi=150)
plt.show()

total_visits = sum(sv.values())
print(f'Total state visits : {total_visits:,}')
print(f'Unique states seen : {len(sv):,}')
print(f'Top state fraction : {counts[0]/total_visits:.1%}  ({labels[0]})')

## 4 — Q-table statistics

In [ ]:
q_max  = np.array([v.max()  for v in Q.values()])
q_min  = np.array([v.min()  for v in Q.values()])
q_mean = np.array([v.mean() for v in Q.values()])
q_span = q_max - q_min

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, data, title, colour in zip(
    axes,
    [q_max, q_mean, q_span],
    ['Max Q per state', 'Mean Q per state', 'Q-value span (max−min) per state'],
    ['steelblue', 'seagreen', 'darkorange']
):
    ax.hist(data, bins=40, color=colour, edgecolor='white', linewidth=0.4)
    ax.axvline(np.median(data), color='k', linestyle='--', linewidth=1, label=f'median={np.median(data):.2f}')
    ax.set_title(title)
    ax.set_xlabel('value')
    ax.set_ylabel('states')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Q-table stats — {len(Q)} states', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(POLICY_DIR, 'qtable_stats.png'), dpi=150)
plt.show()

print(f'States with span < 1e-4 (near-uniform, under-trained): '
      f'{(q_span < 1e-4).sum()} / {len(Q)}')

## 5 — Reward distribution by decile (has the agent improved late in training?)

In [ ]:
n_ep = len(log)
n_bins = 10
bin_size = n_ep // n_bins
decile_means = []
decile_stds  = []
decile_labels = []

for i in range(n_bins):
    chunk = rew[i * bin_size : (i + 1) * bin_size]
    decile_means.append(chunk.mean())
    decile_stds.append(chunk.std())
    lo = i * bin_size + 1
    hi = (i + 1) * bin_size
    decile_labels.append(f'{lo}–{hi}')

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(n_bins)
ax.bar(x, decile_means, yerr=decile_stds, capsize=4,
       color='steelblue', alpha=0.8, edgecolor='white')
ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(decile_labels, rotation=30, fontsize=9)
ax.set_xlabel('Episode range')
ax.set_ylabel('Mean reward ± std')
ax.set_title('Mean reward by training decile')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(POLICY_DIR, 'reward_by_decile.png'), dpi=150)
plt.show()